In [1]:
import os
import pandas as pd
import seaborn as sns

titanic = sns.load_dataset("titanic")
titanic.shape

(891, 15)

In [2]:
manquantes = titanic.isna().sum()
print(manquantes[manquantes > 0])
print()
print((titanic.isna().mean() * 100).round(1)[manquantes > 0])
print()
nb_doublons = titanic.duplicated().sum()
print(f"Lignes en doublon : {nb_doublons}")

age            177
embarked         2
deck           688
embark_town      2
dtype: int64

age            19.9
embarked        0.2
deck           77.2
embark_town     0.2
dtype: float64

Lignes en doublon : 107


In [3]:
titanic[titanic.duplicated(keep=False)].sort_values(list(titanic.columns)).head(6)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
64,0,1,male,NaN,0,0,27.7208,C,First,man,True,NaN,Cherbourg,no,True
295,0,1,male,NaN,0,0,27.7208,C,First,man,True,NaN,Cherbourg,no,True
144,0,2,male,18.0,0,0,11.5000,S,Second,man,True,NaN,Southampton,no,True
757,0,2,male,18.0,0,0,11.5000,S,Second,man,True,NaN,Southampton,no,True
658,0,2,male,23.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
733,0,2,male,23.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True


## Journal de nettoyage

### Décision 1 : doublons conservés
- **Quoi** : aucune ligne supprimée
- **Pourquoi** : 107 lignes sont identiques à une autre, mais le jeu n'a pas d'identifiant. Deux passagers de même classe, sexe, âge, tarif et port sont possibles, et supprimer fausserait les effectifs.
- **Lignes** : 107 examinées, 0 supprimée

In [4]:
titanic = titanic.drop(columns=["deck"])

### Décision 2 : colonne `deck` supprimée
- **Quoi** : colonne supprimée
- **Pourquoi** : 77 % de valeurs manquantes (688 sur 891), trop pour servir
- **Lignes** : 891 (colonne entière)

In [5]:
mediane_par_groupe = titanic.groupby(["pclass", "sex"])["age"].transform("median")
titanic["age"] = titanic["age"].fillna(mediane_par_groupe)
titanic["age"].isna().sum()

np.int64(0)

### Décision 3 : `age` imputé
- **Quoi** : valeurs manquantes remplacées par la médiane de la classe et du sexe du passager
- **Pourquoi** : variable utile à l'analyse, 20 % de manquants, et l'âge dépend de la classe et du sexe. La médiane résiste mieux aux valeurs extrêmes que la moyenne.
- **Lignes** : 177

In [6]:
titanic = titanic.dropna(subset=["embark_town"])
titanic.shape

(889, 14)

### Décision 4 : passagers sans port supprimés
- **Quoi** : 2 lignes supprimées
- **Pourquoi** : très peu de lignes concernées, et choisir un port reviendrait à inventer une information
- **Lignes** : 2

In [7]:
titanic = titanic.drop(columns=["embarked", "class", "alive", "adult_male"])
titanic["pclass"] = titanic["pclass"].astype("category")
titanic.dtypes

survived          int64
pclass         category
sex                 str
age             float64
sibsp             int64
parch             int64
fare            float64
who                 str
embark_town         str
alone              bool
dtype: object

### Décision 5 : colonnes redondantes supprimées, type corrigé
- **Quoi** : suppression de `embarked`, `class`, `alive`, `adult_male` ; `pclass` passe en `category`
- **Pourquoi** : `embarked` répète `embark_town`, `class` répète `pclass`, `alive` répète `survived`, `adult_male` se déduit de `who`. `pclass` vaut 1, 2 ou 3 : c'est une catégorie, pas une quantité.
- **Lignes** : 4 colonnes supprimées, 889 lignes

In [8]:
q1 = titanic["fare"].quantile(0.25)
q3 = titanic["fare"].quantile(0.75)
ecart = q3 - q1
borne_haute = q3 + 1.5 * ecart

billets_hors_bornes = titanic[titanic["fare"] > borne_haute]
print(f"Borne haute : {borne_haute:.2f}")
print(f"Billets au-dessus : {len(billets_hors_bornes)}")
print(billets_hors_bornes["pclass"].value_counts())

Borne haute : 65.66
Billets au-dessus : 114
pclass
1    102
3      7
2      5
Name: count, dtype: int64


### Décision 6 : billets hors bornes conservés
- **Quoi** : aucune ligne supprimée
- **Pourquoi** : 114 billets dépassent 65,66 (règle des 1,5 écart interquartile), dont 102 en 1re classe. Ce sont des prix réels : aberrant ne veut pas dire faux. On signale aussi 15 billets à 0, probablement des passagers invités ou de l'équipage, gardés tels quels.
- **Lignes** : 114 examinées, 0 supprimée

In [9]:
os.makedirs("../data", exist_ok=True)
titanic.to_csv("../data/titanic_propre.csv", index=False)
pd.read_csv("../data/titanic_propre.csv").shape

(889, 10)